# 02 · Generate, evaluate locally, build the Codabench submission
Loads `best.pt` (or `final.pt`) from a run and scores it on our held-out 512→128-word prompts
(BERTScore / ROUGE-L / BLEU-4, approximate local versions of the official metrics). It then
generates the official test continuations and writes `submission.zip` with `prediction.json` at the root.
Every local evaluation is appended to `RUNS_DIR/<run>/local_eval.jsonl` for the report.

In [ ]:
# ---- paths + locate the code (lm_lib/ and models/) ----
# Put the code on Drive in ONE of these ways:
#   (a) upload the folders lm_lib/ and models/ (+ hf_token.txt) into MyDrive/cs781_capstone/   [or any folder]
#   (b) upload cs781_code.zip (made by make_code_zip.py) anywhere in MyDrive -> it is unzipped automatically
from google.colab import drive
drive.mount('/content/drive')
import sys, os, glob, zipfile, importlib, shutil, time
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # less fragmentation; must precede CUDA init
MYDRIVE    = '/content/drive/MyDrive'
DRIVE_ROOT = f'{MYDRIVE}/cs781'            # data + runs live here
DRIVE_DATA = f'{DRIVE_ROOT}/data'
RUNS_DIR   = f'{DRIVE_ROOT}/runs'
LOCAL_DATA = '/content/data'

def find_code_dir():
    levels = ['', '*/', '*/*/', '*/*/*/']
    # 1) a folder on Drive containing lm_lib/ (newest wins if there are several copies)
    hits = [h for lv in levels for h in glob.glob(f'{MYDRIVE}/{lv}lm_lib/trainer.py')]
    if hits:
        return os.path.dirname(os.path.dirname(max(hits, key=os.path.getmtime)))
    # 2) the newest cs781_code*.zip on Drive -> always re-extracted fresh (so an updated zip replaces old code)
    zips = [z for lv in levels for z in glob.glob(f'{MYDRIVE}/{lv}cs781_code*.zip')]
    if zips:
        z = max(zips, key=os.path.getmtime)
        shutil.rmtree('/content/cs781_code', ignore_errors=True)
        with zipfile.ZipFile(z) as f:
            f.extractall('/content/cs781_code')
        print('unzipped', z, '| zip modified', time.ctime(os.path.getmtime(z)))
        return '/content/cs781_code'
    raise FileNotFoundError('Could not find lm_lib/ on Drive. Upload the lm_lib/ and models/ folders '
                            '(e.g. to MyDrive/cs781_capstone/) or upload cs781_code.zip to MyDrive, then re-run.')

CODE_DIR = find_code_dir()
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
importlib.invalidate_caches()
# if lm_lib/models were imported earlier in this kernel, drop them so the fresh code is used
for _m in [m for m in list(sys.modules) if m == 'lm_lib' or m.startswith(('lm_lib.', 'models.')) or m == 'models']:
    del sys.modules[_m]
print('CODE_DIR =', CODE_DIR, '| contents:', sorted(os.listdir(CODE_DIR)))
os.makedirs(DRIVE_DATA, exist_ok=True); os.makedirs(RUNS_DIR, exist_ok=True)

In [ ]:
!pip -q install tokenizers bert-score rouge-score sacrebleu
RUN = 'A_qwen_dense'
WHICH = 'best'            # best | final
if RUN.startswith('C_gdn_hybrid'):
    !pip -q install flash-linear-attention

In [ ]:
# copy tokenised data from Drive to fast local disk (once per runtime); the training file is TRAIN_BIN if set
import shutil, os
os.makedirs(LOCAL_DATA, exist_ok=True)
_train_files = [globals()['TRAIN_BIN']] if 'TRAIN_BIN' in globals() else []
for f in _train_files + ['val.bin', 'tokenizer.json', 'val_prompts.json', 'val_refs.json', 'val_srcs.json', 'test.json']:
    src, dst = f'{DRIVE_DATA}/{f}', f'{LOCAL_DATA}/{f}'
    if os.path.exists(src) and (not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src)):
        shutil.copyfile(src, dst)
!ls -lh {LOCAL_DATA}

In [ ]:
import json, time
from lm_lib.trainer import load_trained
from lm_lib.generate import generate_continuations
from lm_lib.evaluate import local_scores
from lm_lib.submit import write_submission
run_dir = f'{RUNS_DIR}/{RUN}'
model, tok, cfg, mtype = load_trained(run_dir, WHICH)

In [ ]:
# decoding settings: these are experiment variables for the report (temperature 0 = greedy)
SAMPLING = dict(temperature=0.0, top_k=0, top_p=1.0, repetition_penalty=1.1)
GEN = dict(n_words=128, max_new_tokens=256, batch_size=64)

val_prompts = {k: v['text'] for k, v in json.load(open(f'{LOCAL_DATA}/val_prompts.json')).items()}
val_refs = json.load(open(f'{LOCAL_DATA}/val_refs.json'))
preds = generate_continuations(model, tok, mtype, val_prompts, cfg.seq_len, cfg.src_len, sampling=SAMPLING, **GEN)
scores = local_scores(preds, val_refs)
with open(f'{run_dir}/local_eval.jsonl', 'a') as f:
    f.write(json.dumps(dict(time=time.ctime(), ckpt=WHICH, sampling=SAMPLING, gen=GEN, **scores)) + '\n')
for k in list(preds)[:2]:
    print('PROMPT ...', val_prompts[k][-300:], '\nGEN:', preds[k], '\nREF:', val_refs[k]['text'], '\n')

In [ ]:
# official test set -> submission.zip
test = json.load(open(f'{LOCAL_DATA}/test.json'))
test_prompts = {k: v['text'] for k, v in test.items()}
test_preds = generate_continuations(model, tok, mtype, test_prompts, cfg.seq_len, cfg.src_len, sampling=SAMPLING, **GEN)
tag = f"{RUN}_{WHICH}_{time.strftime('%m%d_%H%M')}"
zip_path = write_submission(test_preds, test, f'{run_dir}/submissions/{tag}')
json.dump(dict(sampling=SAMPLING, gen=GEN, ckpt=WHICH), open(f'{run_dir}/submissions/{tag}/settings.json', 'w'))
from google.colab import files
files.download(zip_path)